#  AddCommit 


In [ ]:
options(width = 120)
Sys.setenv(OPENMLS_V8_CHUNK_ROWS = "200000")
script_candidates <- c("statistics_analysis_openmls_v8.R", file.path("statistics", "statistics_analysis_openmls_v8.R"))
script_path <- script_candidates[file.exists(script_candidates)][1]
stopifnot(!is.na(script_path))
source(script_path)
result <- run_openmls_v8_analysis(render_plots = TRUE)
result$plot_registry

In [ ]:
out_dir <- openmls_v8_output_default; plot_dir <- file.path(out_dir, "plots")
registry <- result$plot_registry |> dplyr::filter(filename %in% names(result$plots$objects)) |> dplyr::arrange(plot_kind, suboperation_key, metric_key)
pdf_page_dir <- file.path(out_dir, "plots_pdf_pages"); dir.create(pdf_page_dir, recursive = TRUE, showWarnings = FALSE)
pdf_pages <- character(nrow(registry))
for (i in seq_len(nrow(registry))) {
  fn <- registry$filename[[i]]; po <- result$plots$objects[[fn]]
  pp <- file.path(pdf_page_dir, sub("\\.png$", ".pdf", fn))
  grDevices::pdf(pp, width = registry$width[[i]], height = registry$height[[i]], onefile = FALSE)
  print(po); grDevices::dev.off(); pdf_pages[[i]] <- pp
}
combined_pdf_path <- file.path(plot_dir, "addcommit_all_plots.pdf")
if (requireNamespace("qpdf", quietly = TRUE)) {
  qpdf::pdf_combine(input = pdf_pages, output = combined_pdf_path); cm <- "qpdf"
} else {
  warning("qpdf not installed"); grDevices::pdf(combined_pdf_path, width = 12, height = 6, onefile = TRUE)
  for (i in seq_len(nrow(registry))) print(result$plots$objects[[registry$filename[[i]]]])
  grDevices::dev.off(); cm <- "grDevices fallback"
}
tibble(pdf_page_count = length(pdf_pages), individual_pdf_dir = pdf_page_dir, combine_method = cm, combined_pdf = combined_pdf_path)

In [ ]:
table_dir <- file.path(out_dir, "tables")
list(file_inventory = readr::read_csv(file.path(table_dir, "file_inventory.csv"), show_col_types = FALSE),
  l1d_coverage = readr::read_csv(file.path(table_dir, "l1d_coverage.csv"), show_col_types = FALSE) |> dplyr::filter(op == "add_commit_total_local"),
  metric_missingness = readr::read_csv(file.path(table_dir, "metric_missingness.csv"), show_col_types = FALSE) |> dplyr::filter(missing),
  platform_inventory = result$diagnostics$platform_inventory)

---

# CommitReceive / Process Commit

Schema-10 rows with `operation_family=commit_receive`. Old pre-refactor data skipped. Hard fail on error.

In [ ]:
cr_result <- tryCatch({
  r <- run_openmls_v8_commit_receive_analysis(render_plots = TRUE)
  if (is.null(r)) stop("CommitReceive analysis returned NULL")
  r
}, error = function(e) {
  message("CommitReceive analysis FAILED: ", conditionMessage(e))
  stop(e)
})
if (!is.null(cr_result)) {
  message("CommitReceive: ", nrow(cr_result$data), " rows")
  cr_result$diagnostics$commit_kind_counts
}

In [ ]:
if (!is.null(cr_result)) {
  list(file_inventory = cr_result$file_inventory,
    metadata_coverage = cr_result$diagnostics$metadata_coverage,
    l1d_coverage = cr_result$diagnostics$l1d_coverage)
}

---

# ApplicationMessageCreate / ApplicationMessageReceive

Schema-10 rows with `operation_family=application_message_{create,receive}`. Hard fail on error.

In [ ]:
am_result <- tryCatch({
  r <- run_openmls_v8_app_message_analysis(render_plots = TRUE)
  if (is.null(r)) stop("App message analysis returned NULL")
  r
}, error = function(e) {
  message("App message analysis FAILED: ", conditionMessage(e))
  stop(e)
})
if (!is.null(am_result)) {
  message("App messages: ", nrow(am_result$data), " rows")
  am_result$data |> count(op_family)
}

In [ ]:
if (!is.null(am_result)) {
  list(span_inventory = am_result$diagnostics$span_inventory,
    l1d_coverage = am_result$diagnostics$l1d_coverage,
    platform_inventory = am_result$diagnostics$platform_inventory)
}

---

# UpdateCommitCreate / RemoveCommitCreate

Schema-10 rows. Hard fail on error.

In [ ]:
cc_result <- tryCatch({
  r <- run_openmls_v8_commit_create_analysis(render_plots = TRUE)
  if (is.null(r)) stop("Commit create analysis returned NULL")
  r
}, error = function(e) {
  message("Commit create analysis FAILED: ", conditionMessage(e))
  stop(e)
})
message("Commit create: ", nrow(cc_result$data), " rows")

---

# KeyPackageCreate

Pre-Add key material generation. Schema-10 rows with `operation_family=key_package_create`.
Mostly constant per KeyPackage for fixed config. `artifact_size_bytes` is the key scaling variable.

In [ ]:
kp_result <- tryCatch({
  r <- run_openmls_v8_key_package_analysis(render_plots = TRUE)
  if (is.null(r)) stop("KeyPackage analysis returned NULL")
  r
}, error = function(e) {
  message("KeyPackage analysis FAILED: ", conditionMessage(e))
  stop(e)
})
message("KeyPackage: ", nrow(kp_result$data), " rows")

---

# WelcomeReceive / JoinFromWelcome

New member processes Welcome: decrypts GroupSecrets (HPKE), decrypts GroupInfo (AEAD), imports ratchet tree, initializes group state.
Schema-10 rows with `operation_family=welcome_receive`. Hard fail on error.

Scaling variables: `welcome_bytes`, `ratchet_tree_bytes`, `tree_node_count`, `member_count_after`,
`welcome_recipient_count`. Child spans carry these via context propagation.

In [ ]:
wr_result <- tryCatch({
  r <- run_openmls_v8_welcome_receive_analysis(render_plots = TRUE)
  if (is.null(r)) stop("WelcomeReceive analysis returned NULL")
  r
}, error = function(e) {
  message("WelcomeReceive analysis FAILED: ", conditionMessage(e))
  stop(e)
})
if (!is.null(wr_result)) {
  message("WelcomeReceive: ", nrow(wr_result$data), " rows")
  wr_result$diagnostics$span_inventory
}

In [ ]:
if (!is.null(wr_result)) {
  list(l1d_coverage = wr_result$diagnostics$l1d_coverage,
    platform_inventory = wr_result$diagnostics$platform_inventory)
}

---

# Failure-Experiment Analysis

Failure rows from `events.csv` where `failure_class` is non-null.  Shows **which resource envelopes** (CPU/RAM caps assigned to profiled singletons) survive to **which group sizes**, and **why** failures occur (OOM, container exit, CPU starvation, etc.).

In [ ]:
fe_result <- tryCatch({
  run_openmls_v8_failure_experiment_analysis(render_plots = TRUE)
}, error = function(e) {
  warning("Failure-experiment analysis failed: ", e$message)
  NULL
})
if (!is.null(fe_result) && nrow(fe_result$data) > 0) {
  list(
    failure_events = nrow(fe_result$data),
    resource_profiles = n_distinct(fe_result$data$resource_profile),
    run_count = n_distinct(fe_result$data$source_run_folder),
    heatmap_path = fe_result$plot_paths["heatmap"],
    class_breakdown_path = fe_result$plot_paths["class_breakdown"]
  )
} else {
  "No failure events found in events.csv files."
}

In [ ]:
# Show failure summary table sorted by constraint
if (!is.null(fe_result) && nrow(fe_result$data) > 0) {
  fe_result$summary |>
    dplyr::filter(failure_count > 0) |>
    dplyr::arrange(target_band, profile_short) |>
    print(n = 100)
}